In [ ]:
%pip install numpy
%pip install pandas

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from pandas import DataFrame

T = pd.read_csv('/Users/cathycroome/code/open-plaques-United-Kingdom-2025-12-14.csv')

NameError: name '__file__' is not defined

In [2]:
MONTH_ANGLES = {
    'January':   (60, 90),
    'February':  (30, 60),
    'March':     (0, 30),
    'April':     (330, 360),
    'May':       (300, 330),
    'June':      (270, 300),
    'July':      (240, 270),
    'August':    (210, 240),
    'September': (180, 210),
    'October':   (150, 180),
    'November':  (120, 150),
    'December':  (90, 120),
}

In [3]:
def cartesian_to_polar(x, y):
    """
    Convert Cartesian coordinates to polar coordinates.
    Theta is measured anti-clockwise from the positive x-axis, in the range [0, 2*pi).
    """
    r = np.sqrt(x**2 + y**2)
    t = np.arctan2(y, x)  # radians, range (-pi, pi]

    # if y >= 0: quadrant 1 or 2, angle is already correct
    # if y < 0:  quadrant 3 or 4, add 2*pi to wrap into [0, 2*pi)
    theta = np.where(y >= 0, t, 2 * np.pi + t)

    theta_deg = np.degrees(theta)

    return r, theta_deg

def manual_lat_and_long(location):
    match location:
        case 'Norwich':
            lat, long = 52.633696, 1.289063
        case 'Brighton':
            lat, long = 50.818918, -0.140827
        case 'Aberdeenshire':
            lat, long = 57.581714, -2.627400
        case 'Llangollen':
            lat, long = 52.970465, -3.170968
        case 'York':
            lat, long = 53.960403, -1.081140
        case 'Formby':
            lat, long = 53.558966, -3.075819
        case _:
            raise ValueError(f"Unknown location: {location}")

    return lat, long

def relative_location(lat,long, centre_lat = 53.4153, centre_long=-2.2127):
    """
    Return (x, y) offset of a point from a fixed centre point,
    in degrees of longitude/latitude (not true distance).
    """
    dx_deg = long - centre_long
    dy_deg = lat - centre_lat

    return dx_deg, dy_deg

def degrees_to_km(dx_deg, dy_deg, lat, km_per_degree = 111.0):
    """
    Convert a (dx_deg, dy_deg) offset into approximate (x_km, y_km),
    used to correct for longitude lines converging away from the equator.
    """
    lat_rad = np.radians(lat)
    x_km = dx_deg * km_per_degree * np.cos(lat_rad)
    y_km = dy_deg * km_per_degree
    
    return x_km, y_km

def get_plaques_for_month(month: str, df: DataFrame, max_distance, blue_only = False):
    low, high = MONTH_ANGLES[month]
    df = df[(df['theta_deg'] > low) & (df['theta_deg'] <= high) & (df['km'] < max_distance)]

    if blue_only:
        df = df[df['colour'] == 'blue']

    df = df.sort_values('km', ascending=True)

    return df

def random_plaque(matches: DataFrame):
    if matches.empty:
        return None

    return matches.sample(1).iloc[0]


In [4]:
# test with manual locations 
location = 'York'                               # choose test location
lat, long = manual_lat_and_long(location)       # set latitude and longitude for location
dx_deg, dy_deg = relative_location(lat, long)   # change in lat and long relative to set centre point
x, y = degrees_to_km(dx_deg, dy_deg, lat)       # convert to cartesian (approximate distances in km)
km, theta_deg = cartesian_to_polar(x ,y)        # convert to polar

print('x = ', x)
print ('y = ', y)
print('km = ', km)
print('theta_deg = ', theta_deg)

x =  73.89789344166381
y =  60.50643299999972
km =  95.50878016967316
theta_deg =  39.31007747261311


In [12]:
print(T.columns)

# format data frame
subset = T[['id', 'lead_subject_name', 'latitude', 'longitude',  'area',  'colour']].copy()
subset = subset.dropna(subset=['latitude', 'longitude', 'lead_subject_name', 'colour'])

# calculate r and theta and append to data frame
dx_deg, dy_deg = relative_location(subset.latitude, subset.longitude)   # change in lat and long relative to set centre point
x, y = degrees_to_km(dx_deg, dy_deg, subset.latitude)                   # convert to cartesian (approximate distances in km)
km, theta_deg = cartesian_to_polar(x,y)                                 # convert to polar
subset['km'] = km                                                       # approximate distances in km
subset['theta_deg'] = theta_deg

T.head(5)

Index(['id', 'machine_tag', 'title', 'inscription', 'latitude', 'longitude',
       'country', 'area', 'address', 'erected', 'main_photo', 'colour',
       'organisations', 'language', 'series', 'series_ref', 'geolocated?',
       'photographed?', 'number_of_subjects', 'number_of_male_subjects',
       'number_of_female_subjects', 'number_of_inanimate_subjects',
       'lead_subject_id', 'lead_subject_machine_tag', 'lead_subject_name',
       'lead_subject_surname', 'lead_subject_sex', 'lead_subject_born_in',
       'lead_subject_died_in', 'lead_subject_type', 'lead_subject_roles',
       'lead_subject_primary_role', 'lead_subject_wikipedia',
       'lead_subject_dbpedia', 'lead_subject_image', 'subjects'],
      dtype='str')


,id,machine_tag,title,inscription,latitude,longitude,country,area,address,erected,...,lead_subject_sex,lead_subject_born_in,lead_subject_died_in,lead_subject_type,lead_subject_roles,lead_subject_primary_role,lead_subject_wikipedia,lead_subject_dbpedia,lead_subject_image,subjects
0,9324,openplaques:id=9324,"Charles Dickens, Myles Birket Foster, and Will...",While staying in this house Charles Dickens (E...,50.59723,-1.18632,United Kingdom,"Ventnor, Isle of Wight","Shore Road, Bonchurch",NaN,...,male,1812.0,1870.0,man,"[""novelist"", ""journalist"", ""policeman"", ""son o...",novelist,https://en.wikipedia.org/wiki/Charles_Dickens,http://dbpedia.org/resource/Charles_Dickens,https://commons.wikimedia.org/wiki/Special:Fil...,"[""Charles Dickens|(1812-1870)|man|novelist, jo..."
1,8973,openplaques:id=8973,Henry Maudslay and Maudslay Rope-forming Machi...,Maudslay Rope-forming Machine Designed and man...,51.39432,0.52742,United Kingdom,Chatham,"Chatham Historical Dockyard, The Old Surgery, ...",1986.0,...,male,1771.0,1831.0,man,"[""inventor""]",inventor,https://en.wikipedia.org/wiki/Henry_Maudslay,http://dbpedia.org/resource/Henry_Maudslay,https://commons.wikimedia.org/wiki/Special:Fil...,"[""Henry Maudslay|(1771-1831)|man|inventor"", ""M..."
2,9850,openplaques:id=9850,Kew Bridge Pumping Station grey plaque,Kew Bridge Pumping Station Unique in its appro...,51.48904,-0.29049,United Kingdom,London,"Kew Bridge Steam Museum, Green Dragon Lane, Br...",1997.0,...,object,NaN,NaN,place,"[""pumping station""]",pumping station,NaN,NaN,NaN,"[""Kew Bridge Pumping Station||place|pumping st..."
3,50075,openplaques:id=50075,Blue plaque № 50075,"Meg & Wes August 11, 2018 romped in this disab...",51.56951,0.70367,United Kingdom,Southend-on-Sea,Southend Airport,2018.0,...,NaN,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN,[]
4,51993,openplaques:id=51993,Blue plaque № 51993,Anonymous Maidservant Trade Union Campaigner 1872,56.45907,-2.97129,United Kingdom,Dundee,15 Union Street,NaN,...,NaN,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN,[]


In [6]:
max_distance = 40
month = 'January'

# Get closest results for a given month
matches = get_plaques_for_month(month, subset, max_distance, False)
matches.head(10)

# Get random result for a given month
randomised_result = random_plaque(matches)
print(randomised_result)

id                                       57424
lead_subject_name    Levenshulme South Station
latitude                              53.44075
longitude                             -2.18833
area                                Manchester
colour                                    blue
km                                    3.252166
theta_deg                            60.300581
Name: 16271, dtype: object
